# CS3081 Assignment: Uncertainty and Bayesian Networks
## Exercise 5 part (c): Python with pgmpy

Name: Mohammed Tabahi Student ID: S24108951

Network (Exercise 3): `heatwave → overload`, `heatwave → powercut`, `overload → powercut`, `powercut → cancel`.

### Setup (run once if pgmpy is not installed)

In [1]:
# %pip install pgmpy

### 1. Build the network and check it

In [3]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

# Structure: 4 arrows
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel"),
])

yn = ["yes", "no"]

# P(H = yes) = 0.3
cpd_heatwave = TabularCPD(
    variable="heatwave", variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"heatwave": yn},
)

# P(A = yes | H): H=yes -> 0.6, H=no -> 0.1
cpd_overload = TabularCPD(
    variable="overload", variable_card=2,
    values=[[0.6, 0.1],
            [0.4, 0.9]],
    evidence=["heatwave"], evidence_card=[2],
    state_names={"overload": yn, "heatwave": yn},
)

# P(C = yes | H, A): (yes,yes)=0.7, (yes,no)=0.2, (no,yes)=0.5, (no,no)=0.05
cpd_powercut = TabularCPD(
    variable="powercut", variable_card=2,
    values=[[0.7, 0.2, 0.5, 0.05],
            [0.3, 0.8, 0.5, 0.95]],
    evidence=["heatwave", "overload"], evidence_card=[2, 2],
    state_names={"powercut": yn, "heatwave": yn, "overload": yn},
)

# P(L = yes | C): C=yes -> 0.8, C=no -> 0.1
cpd_cancel = TabularCPD(
    variable="cancel", variable_card=2,
    values=[[0.8, 0.1],
            [0.2, 0.9]],
    evidence=["powercut"], evidence_card=[2],
    state_names={"cancel": yn, "powercut": yn},
)

model.add_cpds(cpd_heatwave, cpd_overload, cpd_powercut, cpd_cancel)
print(model.check_model())   # must print True

C:\Users\Admin.DESKTOP-QKNRRPJ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin.DESKTOP-QKNRRPJ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


True


### 2. Exact inference with Variable Elimination: P(H | L = yes)

In [4]:
infer = VariableElimination(model)
result = infer.query(variables=["heatwave"], evidence={"cancel": "yes"}, show_progress=False)
print(result)

exact = result.get_value(heatwave="yes")
print(f"Exact P(heatwave = yes | cancel = yes) = {exact:.4f}")   # should be 0.5367 (same as Exercise 4)

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
Exact P(heatwave = yes | cancel = yes) = 0.5367


### 3. Rejection sampling with 10,000 samples

In [5]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")],
    size=10000,
    show_progress=False,
)

print("Number of samples kept:", len(samples))
print(samples.head())

estimate = (samples["heatwave"] == "yes").mean()
print(f"Rejection-sampling estimate P(heatwave = yes | cancel = yes) = {estimate:.4f}")
print(f"Exact answer = {exact:.4f}  |  difference = {abs(estimate - exact):.4f}")

Number of samples kept: 10000
  heatwave overload powercut cancel
0       no       no       no    yes
1      yes      yes      yes    yes
2       no       no      yes    yes
3      yes       no      yes    yes
4       no       no       no    yes
Rejection-sampling estimate P(heatwave = yes | cancel = yes) = 0.5417
Exact answer = 0.5367  |  difference = 0.0050


### 4. Comparison

The exact answer from Variable Elimination is P(H = yes | L = yes) = 0.5367, which matches my hand calculation in Exercise 4. The rejection-sampling estimate with 10,000 samples is 0.5417, very close to the exact answer (difference 0.0050), because sampling gives only an approximation, and the error gets smaller when we use more samples.